In [1]:
from pathlib import Path
import json
from PIL import Image

In [16]:
# Dossier contenant les images du volume
DOSSIER_IMAGES = Path("../data/Frêne_volume_1/Images")

# Dossier de sortie du manifest
DOSSIER_SORTIE = Path("../data/Frêne_volume_1/exports/iiif")
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)

# Nom du fichier manifest
FICHIER_MANIFEST = DOSSIER_SORTIE / "manifest_Frene_volume_1.json"

# URL publique GitHub Pages du dépôt
BASE_URL = "https://raphix93.github.io/Projet_Frene"

# URL publique du dossier d'images
BASE_URL_IMAGES = f"{BASE_URL}/data/Fr%C3%AAne_volume_1/Images"

# URL publique du dossier IIIF
BASE_URL_IIIF = f"{BASE_URL}/iiif"

# Métadonnées du volume
TITRE = "Journal de Théophile Rémy Frene, volume 1"
AUTEUR = "Théophile Rémy Frene"
DATE = "1741"
INSTITUTION = "Office des archives de l'État de Neuchâtel"
COTE = "FRENE THEOPHILE-REMY"
LANGUE = "français"
LICENCE = "CC-BY 4.0"
ARK = "https://floraweb.ne.ch/flora/ark:/37964/001136"
DESCRIPTION = "Journal manuscrit de Théophile Rémy Frêne, pasteur neuchâtelois."

In [17]:
def valeur_langue(texte, langue="fr"):
    """Retourne une valeur multilingue conforme à IIIF Presentation API 3."""
    return {langue: [texte]}


def lister_images(dossier):
    """Liste les images dans l'ordre alphabétique."""
    extensions = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
    return sorted(
        [f for f in dossier.iterdir() if f.suffix.lower() in extensions]
    )


def type_image(fichier):
    """Détermine le format MIME de l'image."""
    extension = fichier.suffix.lower()
    if extension in [".jpg", ".jpeg"]:
        return "image/jpeg"
    if extension == ".png":
        return "image/png"
    if extension in [".tif", ".tiff"]:
        return "image/tiff"
    return "application/octet-stream"

In [18]:
def creer_canvas(fichier_image, numero_page):
    """Crée un Canvas IIIF pour une image."""
    
    with Image.open(fichier_image) as img:
        largeur, hauteur = img.size

    nom_image = fichier_image.name
    image_url = f"{BASE_URL_IMAGES}/{nom_image}"

    canvas_id = f"{BASE_URL_IIIF}/canvas/volume1/p{numero_page}"
    annotation_page_id = f"{canvas_id}/annotation-page"
    annotation_id = f"{canvas_id}/annotation/image"

    canvas = {
        "id": canvas_id,
        "type": "Canvas",
        "label": valeur_langue(f"Page {numero_page}"),
        "height": hauteur,
        "width": largeur,
        "items": [
            {
                "id": annotation_page_id,
                "type": "AnnotationPage",
                "items": [
                    {
                        "id": annotation_id,
                        "type": "Annotation",
                        "motivation": "painting",
                        "body": {
                            "id": image_url,
                            "type": "Image",
                            "format": type_image(fichier_image),
                            "height": hauteur,
                            "width": largeur
                        },
                        "target": canvas_id
                    }
                ]
            }
        ]
    }

    return canvas

In [19]:
images = lister_images(DOSSIER_IMAGES)

manifest = {
    "@context": "http://iiif.io/api/presentation/3/context.json",
    "id": f"{BASE_URL_IIIF}/manifest_Frene_volume_1.json",
    "type": "Manifest",
    "label": valeur_langue(TITRE),
    "summary": valeur_langue(DESCRIPTION),
    "metadata": [
        {
            "label": valeur_langue("Auteur"),
            "value": valeur_langue(AUTEUR)
        },
        {
            "label": valeur_langue("Date"),
            "value": valeur_langue(DATE)
        },
        {
            "label": valeur_langue("Institution"),
            "value": valeur_langue(INSTITUTION)
        },
        {
            "label": valeur_langue("Cote"),
            "value": valeur_langue(COTE)
        },
        {
            "label": valeur_langue("Langue"),
            "value": valeur_langue(LANGUE)
        },
        {
            "label": valeur_langue("Licence"),
            "value": valeur_langue(LICENCE)
        },
        {
            "label": valeur_langue("ARK"),
            "value": valeur_langue(ARK)
        }
    ],
    "rights": "https://creativecommons.org/licenses/by/4.0/",
    "requiredStatement": {
        "label": valeur_langue("Attribution"),
        "value": valeur_langue("Office des archives de l'État de Neuchâtel / Raphaël Rollinet")
    },
    "homepage": [
        {
            "id": ARK,
            "type": "Text",
            "label": valeur_langue("Notice d'origine")
        }
    ],
    "items": [
        creer_canvas(fichier_image, numero_page)
        for numero_page, fichier_image in enumerate(images, start=1)
    ]
}

with open(FICHIER_MANIFEST, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(f"Manifest créé : {FICHIER_MANIFEST}")
print(f"Nombre d'images intégrées : {len(images)}")

Manifest créé : ..\data\Frêne_volume_1\exports\iiif\manifest_Frene_volume_1.json
Nombre d'images intégrées : 45


In [11]:

with open(FICHIER_MANIFEST, "r", encoding="utf-8") as f:
    manifest_test = json.load(f)

print("Type :", manifest_test["type"])
print("Titre :", manifest_test["label"]["fr"][0])
print("Nombre de canvas :", len(manifest_test["items"]))

premier_canvas = manifest_test["items"][0]
print("Premier canvas :", premier_canvas["label"]["fr"][0])
print("Image :", premier_canvas["items"][0]["items"][0]["body"]["id"])

Type : Manifest
Titre : Journal de Théophile Rémy Frêne, volume 1
Nombre de canvas : 45
Premier canvas : Page 1
Image : https://raphix93.github.io/Projet_Frene/data/Fr%C3%AAne_volume_1/Images/Image00001.tif
